# CS236781: Deep Learning on Computational Accelerators
# Final Project

Faculty of Computer Science, Technion.

Submitted by:

| #       |              Name |             Id |             email                  |
|---------|-------------------|----------------|----------------------------------- |
|Student 1|  Daniel Elgarici  |   305341828    | elgarici-dan@campus.technion.ac.il |
|Student 2|  Tal Benjo        |   318655701    | tal.benjo@campus.technion.ac.il    |

## Introduction

In this assi

# Setup

In [ ]:
# !pip install torch torchvision transformers datasets


In [1]:
import torch
import torch.nn as nn
from transformers import AutoImageProcessor, AutoModelForImageClassification
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image
import requests

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

cuda


# ============================== #
#       HELPER FUNCTIONS        #
# ============================== #

## Pa

In [12]:
def flip_top_k_scalars(model, n_layers, k, score_fn=abs):
    scalars = []
    mapping = []

    # Step 1: Collect scalars and their source (param tensor + flat index)
    layer_count = 0
    for name, module in model.named_modules():
        if layer_count >= n_layers:
            break
        if isinstance(module, (nn.Sequential, nn.ModuleList, nn.ModuleDict)) or module is model:
            continue
        params = list(module.parameters(recurse=False))
        if not params:
            continue
        for param in params:
            if param.requires_grad and param.ndim >= 2:
                flat = param.detach().view(-1)
                for i, val in enumerate(flat):
                    scalars.append(score_fn(val.item()))
                    mapping.append((param, i))
                    
        layer_count += 1

    # Step 2: Sort scalars by score_fn
    scored = sorted(enumerate(scalars), key=lambda x: x[1], reverse=True)
    top_k_indices = [i for i, _ in scored[:k]]

    # Step 3: Flip sign of top-k scalars
    with torch.no_grad():
        for idx in top_k_indices:
            param, flat_idx = mapping[idx]
            param.view(-1)[flat_idx] *= -1




# ============================== #
#         DNL (Pass-free)       #
# ============================== #


In [4]:
def dnl_passfree(model, k=10, layer_limit=10):
    model.eval()
    flip_top_k_scalars(model, layer_limit, k) 

In [5]:
def single_pass_score(scalar_param):
    g = scalar_param.grad.detach()
    p = scalar_param.detach()
    score = p.abs() + (p * g + 0.5 * (p ** 2) * (g ** 2)).abs()
    return score

# ============================== #
#       1P-DNL (One-pass)       #
# ============================== #

In [10]:
def dnl_1pass(model, k=10, n_layers=10, input_shape=(1, 3, 224, 224)):
    model.eval()

    # Step 1: Forward and backward pass on random input
    dummy_input = torch.randn(input_shape)
    model.zero_grad()
    output = model(dummy_input).logits
    loss = output.sum()  # equivalent to sum_i fθ(X)[i]
    loss.backward()

    scalars = []
    mapping = []

    # Step 2: Extract scalar scores from first n layers
    layer_count = 0
    for name, module in model.named_modules():
        if layer_count >= n_layers:
            break
        if isinstance(module, (nn.Sequential, nn.ModuleList, nn.ModuleDict)) or module is model:
            continue
        params = list(module.parameters(recurse=False))
        if not params:
            continue
        for param in params:
            if param.requires_grad and param.ndim >= 2:
                flat = param.detach().view(-1)
                grad = param.grad.detach().view(-1)
                for i in range(len(flat)):
                    θi = flat[i]
                    gi = grad[i]
                    h_ii = gi ** 2  # Gauss-Newton approx
                    score = abs(θi) + abs(θi * gi + 0.5 * θi**2 * h_ii)
                    scalars.append(score.item())
                    mapping.append((param, i))
        layer_count += 1

    # Step 3: Sort and flip top-k scalars
    topk = sorted(range(len(scalars)), key=lambda i: scalars[i], reverse=True)[:k]

    with torch.no_grad():
        for idx in topk:
            param, i = mapping[idx]
            param.view(-1)[i] *= -1


In [20]:
def get_parms(model, n_layers=10):
    layer_count = 0
    scalars = []
    for name, module in model.named_modules():
        if layer_count >= n_layers:
            break
        if isinstance(module, (nn.Sequential, nn.ModuleList, nn.ModuleDict)) or module is model:
            continue
        params = list(module.parameters(recurse=False))
        if not params:
            continue
        for param in params:
            if param.requires_grad and param.ndim >= 2:
                flat = param.detach().view(-1)
                for i, val in enumerate(flat):
                    scalars.append(val.item())
        layer_count += 1
    return scalars

In [23]:
# Load a CNN model from Hugging Face
model_name = "microsoft/resnet-50"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

# Load test image (dog)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
inputs = processor(images=image, return_tensors="pt")

# Original prediction
with torch.no_grad():
    original = model(**inputs).logits.argmax(dim=-1).item()
    print("Prediction before attack:", original)

org_parms = get_parms(model).copy()

dnl_passfree(model, k=10)  # or dnl_passfree(model, k=25)
# dnl_1pass(model,k=3)

# Prediction after attack
with torch.no_grad():
    attacked = model(**inputs).logits.argmax(dim=-1).item()
    print("Prediction after attack:", attacked)
    print("Prediction changed:", original != attacked)

att_parms = get_parms(model).copy()
for i, (p1, p2) in enumerate(zip(org_parms, att_parms)):
    if p1 != p2:
        print(i, p1, p2)

Prediction before attack: 282
Prediction after attack: 285
Prediction changed: True
1453 -3.1727681159973145 3.1727681159973145
8754 3.181039571762085 -3.181039571762085
11977 4.478638172149658 -4.478638172149658
26371 -2.9016716480255127 2.9016716480255127
27023 -3.778019905090332 3.778019905090332
27412 -3.571671724319458 3.571671724319458
27892 -4.089874267578125 4.089874267578125
28095 -2.894585132598877 2.894585132598877
28136 -4.873111248016357 4.873111248016357
28969 -2.9680492877960205 2.9680492877960205


In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# -------- Settings --------
texts = [
    "I absolutely loved this movie, it was fantastic!",
]

models_to_test = [
    "textattack/bert-base-uncased-SST-2",
    "textattack/roberta-base-SST-2",
]

def predict_label(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
        pred_id = logits.argmax(dim=-1).item()
        label = model.config.id2label.get(pred_id, str(pred_id))
    return pred_id, label

def run_attack_on_model(model_name, text, k=10):
    print(f"\n=== Model: {model_name} ===")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    orig_id, orig_label = predict_label(model, tokenizer, text)
    print("Text:", text)
    print("Prediction before attack:", orig_label, f"({orig_id})")

    org_parms = get_parms(model).copy()

    dnl_passfree(model, k=k)
    # dnl_1pass(model, k=3)

    att_id, att_label = predict_label(model, tokenizer, text)
    print("Prediction after attack :", att_label, f"({att_id})")
    print("Prediction changed      :", orig_id != att_id)

    return model, tokenizer, org_parms

# ---------- Run ----------
for model_name in models_to_test:
    for t in texts:
        _ = run_attack_on_model(model_name, t, k=10)



=== Model: textattack/bert-base-uncased-SST-2 ===


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Some weights of the model checkpoint at textattack/roberta-base-SST-2 were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Text: I absolutely loved this movie, it was fantastic!
Prediction before attack: LABEL_1 (1)
Prediction after attack : LABEL_1 (1)
Prediction changed      : False
